In [ ]:
# %pip install rapidfuzz

In [5]:
import pandas as pd
from rapidfuzz import process, fuzz

import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Sample df1 with words to match
df_ACFull = pd.read_excel(r"path/filename.xlsx",sheet_name="Autocare")
df_ACFull

In [ ]:
# Sample df1 with words to match
df_FBGFull = pd.read_excel(r"path/filename.xlsx",sheet_name="FBG")
df_FBGFull=df_FBGFull[['Part Terminology Name','Cleaned Attributes']]

In [ ]:
df_FBGFull.size

In [9]:
PartNames=list(set(df_FBGFull['Part Terminology Name'].tolist()))

In [10]:
cols =['Part Terminology Name','Cleaned Attributes','approx_matches'] #,'Review_Mentions','Standards'
df_New = pd.DataFrame(columns=cols)
count=0

In [11]:

# Function to get approximate matches with a configurable threshold
def get_matches(word, reference_list, threshold=98):
    results = process.extract(word, reference_list, scorer=fuzz.ratio)
    return [match for match, score, _ in results if score >= threshold]

# Set your desired threshold here
MATCH_THRESHOLD = 45

In [12]:
for i in range (len(PartNames)):
    df_FBGF=df_FBGFull[df_FBGFull['Part Terminology Name']==PartNames[i]].reset_index(drop=True)
    df_ACF=df_ACFull[df_ACFull['PartTerminologyName']==PartNames[i]]

    for j in range(len(df_FBGF)):
        #print(PartNames[i],df_FBGF['Cleaned Attributes'][j])
        df_New.at[count,'Part Terminology Name']=PartNames[i]
        df_New.at[count,'Cleaned Attributes']=df_FBGF['Cleaned Attributes'][j]
        df_New.at[count,'approx_matches']=get_matches(df_FBGF['Cleaned Attributes'][j],df_ACF['PA'].tolist(), threshold=MATCH_THRESHOLD)
        count=count+1  

In [ ]:
df_New = df_New[df_New['approx_matches'].apply(str) != "[]"].reset_index(drop=True)
df_New = df_New.drop_duplicates(subset=['Part Terminology Name', 'Cleaned Attributes'])
df_New

In [ ]:
df_New.to_excel(r'path/filename.xlsx', index=False)

## Misc Code


In [ ]:

# Sample df1 with words to match
df_FB = pd.read_excel(r"path/filename.xlsx",sheet_name="FB")
df_FB

In [ ]:
df_AC = pd.read_excel(r"path/filename.xlsx",sheet_name="AC")
df_AC

In [ ]:

# Function to get approximate matches with a configurable threshold
def get_matches(word, reference_list, threshold=98):
    results = process.extract(word, reference_list, scorer=fuzz.ratio)
    return [match for match, score, _ in results if score >= threshold]

# Set your desired threshold here
MATCH_THRESHOLD = 80


In [ ]:

# Apply the function to df1
df_FB['approx_matches'] = df_FB['Cleaned FBG Attribute'].apply(
    lambda w: get_matches(w, df_AC['AC Attribute'].tolist(), threshold=MATCH_THRESHOLD)
)

df_FB=df_FB.drop_duplicates(inplace=True).rest_index(drop=True)


In [ ]:
df_FB.to_excel(r'path/filename.xlsx', index=False)